In [5]:
import json
import os
import sys
sys.path.append('/home/youssef/Documents/python/music-map/backend')
from data_extraction.db_operations.get_features import execute
result = execute(
    """
select t.id, preview_url, a.name, t.artist_id
from tracks t
join artists a on t.artist_id = a.id
    """
)
import json
result = [{'id': id, 'preview_url': preview_url, 'name': name, 'artist_id': artist_id} for id, preview_url, name, artist_id in result]
with open('../jsons/all_artists_colab.json', 'w') as f:
    json.dump(result, f)

In [1]:
import numpy as np
arr = np.load("../jsons/data.npy")
ids = np.load("../jsons/ids.npy")

In [2]:
# all_songs = np.load("../jsons/full_features.npz")
# arr = all_songs['data']  # assuming all_songs was loaded as npz with key 'data'
# ids = all_songs['ids']
from collections import defaultdict
mert_features  = {}
for i in range(arr.shape[0]):
    mert_features[str(ids[i])] = arr[i][5]

In [ ]:
import sys
sys.path.append("/home/youssef/Documents/python/music-map/backend")
from data_extraction.similarity_model import *
all_tracks = load_all_tracks("/home/youssef/Documents/python/music-map/backend/data_extraction/music.duckdb")
X_pca, pca, scaler, meta = reduce_features(all_tracks, meta_cols=["id", "artist", "artist_id"])
concated = pd.DataFrame(X_pca).join(meta)


In [ ]:
mert_features['2521']

(1024,)

In [ ]:
combined_dict = {}
for i, item in enumerate(concated['id']):
    mert_vec = np.array(mert_features[str(item)]) / np.linalg.norm(mert_features[str(item)])
    essentia_vec = X_pca[i] / np.linalg.norm(X_pca[i])
    combined = np.concatenate([mert_vec, essentia_vec])
    combined_dict[i] = combined
combined_df = pd.DataFrame(combined_dict).T.join(meta)

In [4]:
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity
def get_intra_similarity(song_outputs):
  similarities = []
  for song1, song2 in combinations(song_outputs, 2):
    song1 = np.array(song1).reshape(1, -1)
    song2 = np.array(song2).reshape(1, -1)
    similarities.append(cosine_similarity(song1, song2).item())
  return similarities

from itertools import product
def get_inter_similarity(outputs1, outputs2):
  similarities = []
  for out1, out2 in product(outputs1, outputs2):
    out1 = np.array(out1).reshape(1, -1)
    out2 = np.array(out2).reshape(1, -1)
    similarities.append(cosine_similarity(out1, out2).item())
  return similarities

In [1]:
import pickle

# sorted_ids = sorted(unique_ids)
# with open("../jsons/unique_artists.pkl", "wb") as f:
#     pickle.dump(sorted_ids, f)

# with open("../jsons/combined_df.pkl", "wb") as f:
#     pickle.dump(combined_df, f)

with open("../jsons/combined_df.pkl", "rb") as f:
    combined_df = pickle.load(f)

In [9]:
import numpy as np
from itertools import combinations, product
import tqdm
intra = {}
unique_ids = sorted(list(set(item for item in combined_df['artist_id'])))[:3]
for id in tqdm.tqdm(unique_ids):
    artist_songs = [
        combined_df.iloc[i, :-3].to_numpy() 
        for i in range(combined_df.shape[0]) 
        if id == combined_df.iloc[i]['artist_id']
    ]
    similarities = get_intra_similarity(artist_songs)
    intra[id] = similarities
with open("../jsons/hybrid_full_intra.json", "w") as f:
  json.dump(intra, f)

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:10<00:00,  3.54s/it]


In [ ]:
import numpy as np
from itertools import combinations, product
import tqdm
# intra = {}
unique_ids = sorted(list(set(item for item in combined_df['artist_id'])))
# for id in tqdm.tqdm(unique_ids):
#     artist_songs = [
#         combined_df.iloc[i, :-3].to_numpy() 
#         for i in range(combined_df.shape[0]) 
#         if id == combined_df.iloc[i]['artist_id']
#     ]
#     similarities = get_intra_similarity(artist_songs)
#     intra[id] = similarities
# with open("../jsons/hybrid_full_intra.json", "w") as f:
#   json.dump(intra, f)
import json
from itertools import combinations
import tqdm
from collections import defaultdict

inter = defaultdict(dict)
inter_twoway = defaultdict(dict)
combs = [(a, b) for a, b in combinations(unique_ids, 2)]
for artist1, artist2 in tqdm.tqdm(combs):
  artist1_songs = [combined_df.iloc[song, :-3].to_numpy() for i, song in enumerate(range(combined_df.shape[0])) if artist1 == combined_df.iloc[i]['artist_id']]
  artist2_songs = [combined_df.iloc[song, :-3].to_numpy() for i, song in enumerate(range(combined_df.shape[0])) if artist2 == combined_df.iloc[i]['artist_id']]
  similarities = get_inter_similarity(artist1_songs, artist2_songs)
  inter_twoway[artist1][artist2] = similarities
  inter_twoway[artist2][artist1] = similarities
  inter[artist1][artist2] = similarities
# with open("../jsons/hybrid_full_inter_twoway.json", "w") as f:
#   json.dump(inter_twoway, f)
# with open("../jsons/hybrid_full_inter.json", "w") as f:
#   json.dump(inter, f)
# import json
# import numpy as np
# with open("../jsons/hybrid_full_intra.json", "r") as f:
#     intra = json.load(f)

# intra_sims = []
# for artist_id, value in intra.items():
#     intra_sims += value

# import json
# with open("../jsons/hybrid_full_inter.json", "r") as f:
#     inter = json.load(f)

# inter_sims = []
# for artist_id, value in inter.items():
#     for artist_id2, value2 in value.items():
#         inter_sims +=value2
# import numpy as np
# avgs = {}
# avgs_twoway = {}
# medians = {}
# for key, value in inter.items():
#     avgs[key] = {}
#     medians[key] = {}
#     for other, sim in value.items():
#         avgs[key][other] = np.mean(sim).item()
#         medians[key][other] = np.median(sim).item()
# for key, value in inter_twoway.items():
#     avgs_twoway[key] = {}
#     for other, sim in value.items():
#         avgs_twoway[key][other] = np.mean(sim).item()

# with open("../jsons/hybrid_medians.json", "w") as f:
#     json.dump(medians, f)
# with open("../jsons/hybrid_full_avgs_twoway.json", "w") as f:
#     json.dump(avgs_twoway, f)

# with open("../jsons/hybrid_full_avgs.json", "w") as f:
#     json.dump(avgs, f)
# # avgs
# with open("../jsons/hybrid_full_avgs_twoway.json", "r") as f:
#     avgs_twoway = json.load(f)

# from data_extraction.similarity_model import get_names, lookup_artist
# lookup_table = dict(get_names())
# inter_names_twoway = defaultdict(dict)
# for id, value in avgs_twoway.items():
#     for other_id, sim in value.items():
#         name = lookup_artist(id)
#         othername = lookup_artist(other_id)
#         inter_names_twoway[name][othername] = sim 
# inter_names_twoway
# with open("../jsons/hybrid_full_avgs_twoway_named.json", "w") as f:
#     json.dump(inter_names_twoway, f)
# with open("../jsons/hybrid_full_avgs.json", "r") as f:
#     avgs = json.load(f)

# from data_extraction.similarity_model import get_names, lookup_artist
# lookup_table = dict(get_names())
# inter_names = defaultdict(dict)
# for id, value in avgs.items():
#     for other_id, sim in value.items():
#         name = lookup_artist(id)
#         othername = lookup_artist(other_id)
#         inter_names[name][othername] = sim 
# inter_names
# with open("../jsons/hybrid_full_avgs_named.json", "w") as f:
#     json.dump(inter_names, f)
# import json

# # with open('../jsons/hybrid_named.json', 'r') as f:
# #     data = json.load(f)
# # with open('../jsons/artistNames.json', 'r') as f:
# #     reversed = json.load(f)

# with open("../jsons/hybrid_full_avgs_twoway_named.json", "r") as f:
#     inter_names_twoway = json.load(f)

# sorted(inter_names_twoway['عمرو دياب'].items(), key = lambda x: x[1], reverse=True) 

  0%|          | 21/653796 [02:45<1435:21:13,  7.90s/it]


KeyboardInterrupt: 

In [4]:
import sys
# sys.path.append("/home/youssef/Documents/python/music-map/backend")
from data_extraction.similarity_model import *
json.dump(dict(get_names()), open("../jsons/artist_lookup.json", "w"))

In [3]:
import numpy as np
import json
from itertools import combinations
from collections import defaultdict

# --- 1. Pull out the feature matrix once ---
features = combined_df.iloc[:, :-3].to_numpy(dtype=np.float64)
unique_ids = sorted(list(set(item for item in combined_df['artist_id'])))
# --- 2. Normalize rows once so cosine similarity = dot product ---
norms = np.linalg.norm(features, axis=1, keepdims=True)
norms[norms == 0] = 1  # avoid div by zero
features_norm = features / norms

# --- 3. Group row indices by artist_id ONCE (no more per-pair scanning) ---
artist_to_indices = combined_df.groupby('artist_id').indices  # dict: artist_id -> np.array of row idx

# Precompute each artist's normalized song matrix once
artist_matrices = {
    artist_id: features_norm[idx]
    for artist_id, idx in artist_to_indices.items()
}

# --- 4. Compute pairwise similarities via matmul instead of sklearn per-pair calls ---
inter = defaultdict(dict)
inter_twoway = defaultdict(dict)

combs = list(combinations(unique_ids, 2))
import tqdm
for artist1, artist2 in tqdm.tqdm(combs):
    A = artist_matrices[artist1]   # shape (5, d)
    B = artist_matrices[artist2]   # shape (5, d)
    sim_matrix = A @ B.T           # shape (5, 5), cosine similarities since rows are normalized
    similarities = sim_matrix.flatten().tolist()

    inter_twoway[artist1][artist2] = similarities
    inter_twoway[artist2][artist1] = similarities
    inter[artist1][artist2] = similarities

with open("../jsons/hybrid_full_inter_twoway.json", "w") as f:
    json.dump(inter_twoway, f)
with open("../jsons/hybrid_full_inter.json", "w") as f:
    json.dump(inter, f)

100%|██████████| 653796/653796 [00:20<00:00, 31641.57it/s]


In [5]:
import sys
sys.path.append("/home/youssef/Documents/python/music-map/backend")
import json
import numpy as np
from tqdm import tqdm
# with open("../jsons/hybrid_full_intra.json", "r") as f:
#     intra = json.load(f)

# intra_sims = []
# for artist_id, value in intra.items():
#     intra_sims += value

# import json
# with open("../jsons/hybrid_full_inter.json", "r") as f:
#     inter = json.load(f)

# inter_sims = []
# for artist_id, value in tqdm(inter.items()):
#     for artist_id2, value2 in value.items():
#         inter_sims +=value2


# import numpy as np
# avgs = {}
# avgs_twoway = {}
# medians = {}
# for key, value in tqdm(inter.items()):
#     avgs[key] = {}
#     medians[key] = {}
#     for other, sim in value.items():
#         avgs[key][other] = np.mean(sim).item()
#         medians[key][other] = np.median(sim).item()
# for key, value in tqdm(inter_twoway.items()):
#     avgs_twoway[key] = {}
#     for other, sim in value.items():
#         avgs_twoway[key][other] = np.mean(sim).item()

# with open("../jsons/hybrid_medians.json", "w") as f:
#     json.dump(medians, f)
# with open("../jsons/hybrid_full_avgs_twoway.json", "w") as f:
#     json.dump(avgs_twoway, f)

# with open("../jsons/hybrid_full_avgs.json", "w") as f:
#     json.dump(avgs, f)
# avgs
# with open("../jsons/hybrid_full_avgs_twoway.json", "r") as f:
#     avgs_twoway = json.load(f)

from data_extraction.similarity_model import get_names, lookup_artist
lookup_table = dict(get_names())
inter_names_twoway = defaultdict(dict)
for id, value in tqdm(avgs_twoway.items()):
    for other_id, sim in value.items():
        name = lookup_artist(id)
        othername = lookup_artist(other_id)
        inter_names_twoway[name][othername] = sim 
# inter_names_twoway
with open("../jsons/hybrid_full_avgs_twoway_named.json", "w") as f:
    json.dump(inter_names_twoway, f)
# with open("../jsons/hybrid_full_avgs.json", "r") as f:
#     avgs = json.load(f)

from data_extraction.similarity_model import get_names, lookup_artist
lookup_table = dict(get_names())
inter_names = defaultdict(dict)
for id, value in tqdm(avgs.items()):
    for other_id, sim in value.items():
        name = lookup_artist(id)
        othername = lookup_artist(other_id)
        inter_names[name][othername] = sim 
# inter_names
with open("../jsons/hybrid_full_avgs_named.json", "w") as f:
    json.dump(inter_names, f)
import json

# with open('../jsons/hybrid_named.json', 'r') as f:
#     data = json.load(f)
# with open('../jsons/artistNames.json', 'r') as f:
#     reversed = json.load(f)

# with open("../jsons/hybrid_full_avgs_twoway_named.json", "r") as f:
#     inter_names_twoway = json.load(f)

sorted(inter_names_twoway['عمرو دياب'].items(), key = lambda x: x[1], reverse=True) 

100%|██████████| 1143/1143 [00:00<00:00, 3020.84it/s]


[('علاء عبد الخالق', 0.5084466919599369),
 ('عادل عكله', 0.5076081179615654),
 ('Jana Diab', 0.5071814478701175),
 ('صابر الرباعي', 0.5044064335448996),
 ('جواد العلي', 0.5023464806418365),
 ('صباح محمود', 0.5004075550980494),
 ('شفيق جلال', 0.5002671827570755),
 ('Abu Sultan', 0.5002048244987032),
 ('خالد سليم', 0.5000328722885459),
 ('سمية بعلبكي', 0.5000203419576685),
 ('Aziz Maraka', 0.4999926137294166),
 ('بهاء سلطان', 0.49925121351648855),
 ('أحمد الشريف', 0.498962507594256),
 ('وردة', 0.49860612158695444),
 ('سالم طربيه', 0.49851070621170684),
 ('راغب علامة', 0.49798650077132417),
 ('طلعت زين', 0.4978151843197562),
 ('DuOud', 0.49722833816883777),
 ('الشاب عزيز', 0.4970195454402307),
 ('علي كاكولي', 0.49658976461296034),
 ('رامي صبري', 0.49582828466214407),
 ('فهد الكبيسي', 0.4957946190422204),
 ('Arnabeat', 0.49573629199926644),
 ('محمد محي', 0.49517223978991565),
 ('L’Algérino', 0.4944328673049114),
 ('راشد الماجد', 0.4942940821280061),
 ('وديع مراد', 0.4940966715893266),
 ('K

In [ ]:
import numpy as np
def cohens_d(intra_sims, inter_sims):
    n1, n2 = len(intra_sims), len(inter_sims)
    var1, var2 = np.var(intra_sims, ddof=1), np.var(inter_sims, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (np.mean(intra_sims) - np.mean(inter_sims)) / pooled_std
cohens_d(intra_sims, inter_sims)

np.float64(1.9449909568088999)

In [7]:
sorted(inter_names_twoway['فريد الأطرش'].items(), key = lambda x: x[1], reverse=True) 

[('أسمهان', 0.48521901486982194),
 ('محمد قنديل', 0.4825530532669109),
 ('Amr Ismail', 0.4804041853256979),
 ('Oulaya', 0.47807734224333964),
 ('عبد الحليم حافظ', 0.47684578614791817),
 ('Marwan Al shami', 0.47502647470517817),
 ('Takfarinas', 0.47492629543977577),
 ('Natacha Atlas', 0.4746927543202421),
 ('محمود ﯕينيا', 0.47465465892628067),
 ('Clarissa Bitar', 0.47459052283282943),
 ('Al-Walid', 0.4743750086666433),
 ('وليد توفيق', 0.47347609191882106),
 ('Nour El Houda', 0.4734610202828317),
 ('نور الهدى', 0.4734610202828316),
 ('Issaf', 0.473281953521834),
 ('الصواريخ', 0.47323962658180063),
 ('فارس', 0.47295344823501984),
 ('محمد فوزي', 0.47283988050490705),
 ('دارين حدشيتي', 0.47274114702886244),
 ('شادية', 0.47272320644833876),
 ('مي حريري', 0.4726870949648213),
 ('الشيخ إمام', 0.4726391212048064),
 ('وائل جسار', 0.4726144213893733),
 ('محمد عبد المطلب', 0.47248159364537196),
 ('Toni Qattan', 0.4723811630597981),
 ('عايض', 0.4721869043991716),
 ('كارول سماحة', 0.4719636628722841